In [0]:
from pyspark.sql import functions as F

customers_initial_path = (
    "/Volumes/workspace/revenue_leakage_bronze/"
    "landing/customers/initial_load"
)

customers_initial_df = spark.read.json(customers_initial_path)

print(f"Initial customers loaded: {customers_initial_df.count()}")
display(customers_initial_df.limit(10))

In [0]:
# Preserve the original data types
signup_date_type = customers_initial_df.schema["signup_date"].dataType
event_timestamp_type = customers_initial_df.schema["event_timestamp"].dataType


# 1. UPDATE events — modify the first 250 existing customers
customer_updates_df = (
    customers_initial_df
    .orderBy("customer_id")
    .limit(250)
    .withColumn(
        "customer_segment",
        F.when(F.col("customer_segment") == "Standard", "Premium")
         .when(F.col("customer_segment") == "Premium", "VIP")
         .otherwise("Standard")
    )
    .withColumn(
        "customer_status",
        F.when(
            F.regexp_extract("customer_id", r"(\d+)$", 1).cast("int") % 10 == 0,
            "Inactive"
        ).otherwise(F.col("customer_status"))
    )
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn(
        "event_timestamp",
        F.lit("2026-08-16 10:00:00").cast(event_timestamp_type)
    )
)


# 2. INSERT events — create 200 new customers
customer_inserts_df = (
    customers_initial_df
    .orderBy("customer_id")
    .limit(200)
    .withColumn(
        "_new_id_number",
        F.regexp_extract("customer_id", r"(\d+)$", 1).cast("int") + 5000
    )
    .withColumn(
        "customer_id",
        F.format_string("C%06d", F.col("_new_id_number"))
    )
    .withColumn(
        "email",
        F.concat(
            F.lower("first_name"),
            F.lit("."),
            F.lower("last_name"),
            F.col("_new_id_number").cast("string"),
            F.lit("@example.com")
        )
    )
    .withColumn(
        "signup_date",
        F.lit("2026-08-16").cast(signup_date_type)
    )
    .withColumn("customer_status", F.lit("Active"))
    .withColumn("operation", F.lit("INSERT"))
    .withColumn(
        "event_timestamp",
        F.lit("2026-08-16 10:05:00").cast(event_timestamp_type)
    )
    .drop("_new_id_number")
)


# 3. DELETE events — remove the last 50 existing customers
customer_deletes_df = (
    customers_initial_df
    .orderBy(F.desc("customer_id"))
    .limit(50)
    .withColumn("operation", F.lit("DELETE"))
    .withColumn(
        "event_timestamp",
        F.lit("2026-08-16 10:10:00").cast(event_timestamp_type)
    )
)


# Combine all change events into a single CDC batch
customer_changes_df = (
    customer_updates_df
    .unionByName(customer_inserts_df)
    .unionByName(customer_deletes_df)
)


# Validate the generated CDC batch
print(f"Total CDC events: {customer_changes_df.count()}")

display(
    customer_changes_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

In [0]:
customer_changes_path = (
    "/Volumes/workspace/revenue_leakage_bronze/"
    "landing/customers/change_batch_001"
)

# Save the first customer CDC batch as raw JSON files
(
    customer_changes_df.write
    .format("json")
    .mode("overwrite")
    .save(customer_changes_path)
)

# Read the saved files back for validation
saved_customer_changes_df = spark.read.json(customer_changes_path)

print(f"Saved CDC events: {saved_customer_changes_df.count()}")

display(
    saved_customer_changes_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)